# Prepare data for Batch Correction

In [29]:
import pandas as pd
import numpy as np

In [30]:
clinical_gex = pd.read_csv('../dataset/created/clinical_gex_train.csv', low_memory=False)

print(clinical_gex.shape)
clinical_gex.head()

(93, 51486)


,patientID,sex,age,AJCC_stage,M_stage,LDH,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label,...,sept-05,sept-06,sept-07,sept-08,sept-09,sept-10,sept-11,sept-12,sept-14,sept-15
0,YR_106008,male,55,IV,M1C,elevated,no,no,Yan et al.,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RL_WMD-009,female,59,IV,M1C,elevated,no,no,Rizos et al.,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RL_MTP-095,female,55,IV,M1C,elevated,no,no,Rizos et al.,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,YR_2296,male,36,IV,M1C,elevated,no,no,Yan et al.,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,YR_2346,female,64,IV,M1C,normal,no,no,Yan et al.,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Metadata (not used as features)
_meta_cols = ['patientID', 'sample_id', 'pfs_label', 'source']

# Clinical feature groups
clinical_ordinal_col    = ['M_stage']
clinical_categorical_cols = ['sex', 'AJCC_stage', 'LDH', 'immunotherapy_treatment', 'pre_MAPKi_treatment']
clinical_numeric_col    = ['age']
drug_braf_cols          = ['cobimetinib', 'trametinib', 'dabrafenib', 'vemurafenib',
                           'C195W', 'K601I', 'MND', 'V600E', 'V600K', 'V600R']

all_non_gex  = clinical_ordinal_col + clinical_categorical_cols + clinical_numeric_col + drug_braf_cols
CLINICAL_COLS = _meta_cols + all_non_gex
GEX_COLS = [c for c in clinical_gex.columns if c not in CLINICAL_COLS]

print(f'clinical + gex = {len(CLINICAL_COLS)} + {len(GEX_COLS)} = {len(CLINICAL_COLS)+len(GEX_COLS)}')

clinical + gex = 21 + 51465 = 51486


In [32]:
gex = clinical_gex.iloc[:, len(CLINICAL_COLS)-2:]

print(gex.shape)
gex.head()

(93, 51467)


,V600R,sample_id,7A5,A1BG,A1BG-AS1,A1CF,A26C3,A2BP1,A2LD1,A2M,...,sept-05,sept-06,sept-07,sept-08,sept-09,sept-10,sept-11,sept-12,sept-14,sept-15
0,0,05420145C,NaN,NaN,1.218713,0.005668,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,28088 PreB,9.254738,6.637868,NaN,0.182197,-0.950907,1.200617,60.28968,4732.1400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,78853 PreB,0.582902,10.156350,NaN,5.064795,-2.095152,3.217565,164.32100,663.4597,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,05320188B,NaN,NaN,0.680876,0.024512,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,05320143B,NaN,NaN,1.208135,0.025763,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
gex_mat = gex.set_index('sample_id').T

print(gex_mat.shape)
gex_mat.head()

(51466, 93)


sample_id,05420145C,28088 PreB,78853 PreB,05320188B,05320143B,28067_028D PreB,30509 PreB,05320205B,56241 PreB,34A,...,25A,22A,30230_035L PreB,04240151B,56216 PreB,29483_102I PreC,2991 PreB,05320243B,05420180C,05320302B
V600R,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,...,0.0,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7A5,NaN,9.254738,0.582902,NaN,NaN,6.455346,-0.038203,NaN,-2.187514,NaN,...,NaN,NaN,-0.751382,NaN,-0.032107,-3.190759,3.011999,NaN,NaN,NaN
A1BG,NaN,6.637868,10.156350,NaN,NaN,26.215470,19.500620,NaN,25.907530,19.71,...,6.2,6.94,40.428450,NaN,16.689200,10.440390,30.684150,NaN,NaN,NaN
A1BG-AS1,1.218713,NaN,NaN,0.680876,1.208135,NaN,NaN,2.115146,NaN,NaN,...,NaN,NaN,NaN,1.794694,NaN,NaN,NaN,0.555565,0.989007,0.746464
A1CF,0.005668,0.182197,5.064795,0.024512,0.025763,7.869512,-1.160469,0.000000,1.983719,0.00,...,0.0,0.00,2.272507,0.008215,10.062450,7.424451,15.282810,0.000000,0.000000,0.008296


In [34]:
sample_source = clinical_gex[['sample_id', 'source', 'patientID', 'pfs_label']]

print(sample_source.shape)
sample_source.head()

(93, 4)


,sample_id,source,patientID,pfs_label
0,05420145C,Yan et al.,YR_106008,0
1,28088 PreB,Rizos et al.,RL_WMD-009,1
2,78853 PreB,Rizos et al.,RL_MTP-095,0
3,05320188B,Yan et al.,YR_2296,0
4,05320143B,Yan et al.,YR_2346,0


In [35]:
sample_source.to_csv(f"../dataset/created/sample_source.csv", index=False)
gex_mat.index.name = 'sample_id'
gex_mat.to_csv(f"../dataset/created/gex_mat.csv", index=True)